### Draw main graph

In [1]:
import numpy as np
import tensorflow.compat.v1 as tf
from config import *
from GPT_Model import *
from data_pipeline import get_data, data_train_files, data_test_files

tf.reset_default_graph()

tf.compat.v1.disable_eager_execution()
X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])

X_onehot = tf.one_hot(X, axis=2, depth=hparams.n_vocab[0])

logits = model(hparams, X)['logits']
probs = tf.nn.softmax(logits, axis=2)
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

temperature = 0
u = tf.random.uniform(shape=tf.shape(logits[:, -1]), minval=1e-5, maxval=1.-1e-5)
u = (logits[:, -1] - tf.log(temperature + 1e-8)) - tf.log(-tf.log(u))
sample = tf.argmax(u, axis=1)

'''
Train
'''
global_step = tf.Variable(0, name='global_step')
learning_rate = tf.Variable(1e-3, name='learning_rate')

if mode == "pretrain":
    train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)
elif mode == "finetune":
    optimizer = tf.train.AdamOptimizer(learning_rate)
    output_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='liner1|liner2')
    train_step = optimizer.minimize(loss, var_list=output_vars, global_step=global_step)

'''
Session Open
'''

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
graph create


### Load model if exist && TensorboardX Logger

In [2]:
import tf_slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = save_dir ='../save_model'

#只恢复transformer部分的参数
sess.run(tf.global_variables_initializer())
ref_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='transformer')
saver = tf.train.Saver(ref_vars)

restore_file = tf.train.latest_checkpoint(load_dir)
print(restore_file)
if restore_file is not None:
    saver.restore(sess, restore_file)
    print("Model restored.", restore_file)
else:
    print('model not exist.')

#Logger
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)  
        

../save_model/checkpoint-100
INFO:tensorflow:Restoring parameters from ../save_model/checkpoint-100
Model restored. ../save_model/checkpoint-100


### Train

In [3]:
from IPython.display import clear_output
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
from time import sleep
import time
import math

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train', 0, type)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
#         print(_inputs.shape, _targets.shape)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets,
                                                     learning_rate: 1e-5})
        
        train_perplexity = math.exp(_loss) #log perplexity和交叉熵等价
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
            print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 100 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
0	 5.9817038	 396.11467720758384	
Model saved in path: ../save_model/checkpoint-0
10	 4.5672674	 96.28065518694025	
20	 4.6192226	 101.41516548715038	
30	 4.869099	 130.20356888205032	
40	 4.5288506	 92.65200151071338	
50	 4.7156053	 111.67638385205892	
60	 4.548001	 94.4434093751185	
70	 4.6016426	 99.64786372458195	
80	 4.4815006	 88.36717928172715	
90	 4.5102563	 90.94512388936381	
100	 4.350667	 77.53015828432623	
Model saved in path: ../save_model/checkpoint-100
110	 4.6228147	 101.78010525515629	
120	 4.9223776	 137.3287362707861	
130	 4.721087	 112.29024348180457	
140	 4.7257295	 112.81276138206165	
150	 4.534812	 93.2059884676207	
160	 4.49366	 89.44822561492701	
170	 4.6465816	 104.22808781698758	
180	 4.404271	 81.7994995642438	
190	 4.64222	 103.77447348383484	
200	 4.5210457	 91.9316792950207	
Model saved in path: ../save_model/checkpoint-200
210	 4.2121034	 67.49836442551518	
220	 4.1219726	 61.680791459861695	
230	 4.2415333	 69.51435509

KeyboardInterrupt: 

### Compute perplexity on test set

In [4]:
inputs = []
targets = []

l = len(data_test_files)

for i in range(l): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files,'test',i, 'music')
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
    
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity

35.28789480801922